# PySpark SQL Business Project — Hospital Analytics

**Input files:** `patients.csv`, `visits.csv`

**Scenarios:** 21

This notebook uses Spark SQL for business analysis. All SQL is self-contained after the setup cell.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

spark = SparkSession.builder.appName('PySparkSQLProject').getOrCreate()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('PySparkSQLProject').getOrCreate()

entities = spark.read.option('header', True).option('inferSchema', True).csv('patients.csv')
transactions = spark.read.option('header', True).option('inferSchema', True).csv('visits.csv')

transactions = transactions.withColumn('visit_date', to_date(col('visit_date')))

entities.createOrReplaceTempView('entities')
transactions.createOrReplaceTempView('transactions')

print('Entities:', entities.count())
print('Transactions:', transactions.count())
entities.printSchema()
transactions.printSchema()

## Q1. Basic SELECT

Display sample transaction records.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions LIMIT 10
""")
result.show(20, truncate=False)

## Q2. Column Selection

Select important business columns.

In [ ]:
result = spark.sql("""
SELECT patient_id, city, treatment_cost FROM transactions LIMIT 20
""")
result.show(20, truncate=False)

## Q3. WHERE Filter

Find transactions above the overall average.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE treatment_cost > (SELECT AVG(treatment_cost) FROM transactions)
""")
result.show(20, truncate=False)

## Q4. BETWEEN Filter

Filter values between minimum and average.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE treatment_cost BETWEEN (SELECT MIN(treatment_cost) FROM transactions) AND (SELECT AVG(treatment_cost) FROM transactions)
""")
result.show(20, truncate=False)

## Q5. IN Filter

Keep the three most frequent categories.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE department IN (SELECT department FROM transactions GROUP BY department ORDER BY COUNT(*) DESC LIMIT 3)
""")
result.show(20, truncate=False)

## Q6. LIKE Search

Find entity records whose category/name contains the letter a.

In [ ]:
result = spark.sql("""
SELECT * FROM entities WHERE city LIKE '%a%'
""")
result.show(20, truncate=False)

## Q7. CASE WHEN

Create a business value segment.

In [ ]:
result = spark.sql("""
SELECT patient_id, treatment_cost,
CASE WHEN treatment_cost < (SELECT AVG(treatment_cost) FROM transactions) * 0.7 THEN 'Low'
     WHEN treatment_cost <= (SELECT AVG(treatment_cost) FROM transactions) * 1.3 THEN 'Medium'
     ELSE 'High' END AS value_segment
FROM transactions
""")
result.show(20, truncate=False)

## Q8. GROUP BY Count

Count records by category.

In [ ]:
result = spark.sql("""
SELECT department, COUNT(*) AS record_count FROM transactions GROUP BY department ORDER BY record_count DESC
""")
result.show(20, truncate=False)

## Q9. GROUP BY Aggregation

Calculate major KPIs by category.

In [ ]:
result = spark.sql("""
SELECT department, SUM(treatment_cost) AS total_value, AVG(treatment_cost) AS avg_value, MIN(treatment_cost) AS min_value, MAX(treatment_cost) AS max_value FROM transactions GROUP BY department ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q10. HAVING

Show categories whose total is above the average category total.

In [ ]:
result = spark.sql("""
SELECT department, SUM(treatment_cost) AS total_value FROM transactions GROUP BY department HAVING SUM(treatment_cost) > (SELECT AVG(total_value) FROM (SELECT SUM(treatment_cost) AS total_value FROM transactions GROUP BY department)) ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q11. INNER JOIN

Join entity and transaction data.

In [ ]:
result = spark.sql("""
SELECT e.*, t.visit_date, t.treatment_cost, t.department
FROM entities e
INNER JOIN transactions t ON e.patient_id = t.patient_id
LIMIT 50
""")
result.show(20, truncate=False)

## Q12. JOIN Aggregation

Calculate joined business KPIs.

In [ ]:
result = spark.sql("""
SELECT e.city, COUNT(t.patient_id) AS transaction_count, SUM(t.treatment_cost) AS total_value
FROM entities e
JOIN transactions t ON e.patient_id=t.patient_id
GROUP BY e.city
ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q13. LEFT JOIN

Find transaction coverage for every entity.

In [ ]:
result = spark.sql("""
SELECT e.patient_id, e.city, COUNT(t.patient_id) AS transaction_count
FROM entities e
LEFT JOIN transactions t ON e.patient_id=t.patient_id
GROUP BY e.patient_id, e.city
ORDER BY transaction_count DESC
""")
result.show(20, truncate=False)

## Q14. CTE Analysis

Calculate category contribution percentage.

In [ ]:
result = spark.sql("""
WITH category_sales AS (
    SELECT department, SUM(treatment_cost) AS total_value
    FROM transactions
    GROUP BY department
)
SELECT department, total_value,
       ROUND(total_value * 100.0 / SUM(total_value) OVER (), 2) AS contribution_pct
FROM category_sales
ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q15. ROW_NUMBER

Rank each transaction inside its category.

In [ ]:
result = spark.sql("""
SELECT *,
ROW_NUMBER() OVER (PARTITION BY department ORDER BY treatment_cost DESC) AS row_num
FROM transactions
""")
result.show(20, truncate=False)

## Q16. DENSE_RANK

Create dense ranking inside categories.

In [ ]:
result = spark.sql("""
SELECT *,
DENSE_RANK() OVER (PARTITION BY department ORDER BY treatment_cost DESC) AS category_rank
FROM transactions
""")
result.show(20, truncate=False)

## Q17. LAG Comparison

Compare each value with the previous record.

In [ ]:
result = spark.sql("""
SELECT *,
LAG(treatment_cost) OVER (PARTITION BY department ORDER BY visit_date) AS previous_value,
treatment_cost - LAG(treatment_cost) OVER (PARTITION BY department ORDER BY visit_date) AS change_value
FROM transactions
""")
result.show(20, truncate=False)

## Q18. Running Total

Calculate running totals by category.

In [ ]:
result = spark.sql("""
SELECT *,
SUM(treatment_cost) OVER (
    PARTITION BY department
    ORDER BY visit_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS running_total
FROM transactions
""")
result.show(20, truncate=False)

## Q19. Monthly Trend

Analyze monthly business performance.

In [ ]:
result = spark.sql("""
SELECT YEAR(visit_date) AS year_value,
MONTH(visit_date) AS month_value,
SUM(treatment_cost) AS total_value
FROM transactions
GROUP BY YEAR(visit_date), MONTH(visit_date)
ORDER BY year_value, month_value
""")
result.show(20, truncate=False)

## Q20. Top 10

Find the top ten transactions.

In [ ]:
result = spark.sql("""
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (ORDER BY treatment_cost DESC) AS rn
    FROM transactions
) x
WHERE rn <= 10
ORDER BY treatment_cost DESC
""")
result.show(20, truncate=False)

## Q21. Final Business Report

Create the final management KPI report.

In [ ]:
result = spark.sql("""
WITH category_kpi AS (
    SELECT department,
           COUNT(*) AS records,
           SUM(treatment_cost) AS total_value,
           AVG(treatment_cost) AS average_value,
           MIN(treatment_cost) AS minimum_value,
           MAX(treatment_cost) AS maximum_value
    FROM transactions
    GROUP BY department
)
SELECT *,
       DENSE_RANK() OVER (ORDER BY total_value DESC) AS business_rank
FROM category_kpi
ORDER BY business_rank
""")
result.show(20, truncate=False)

## Project Complete

All scenarios use Spark SQL and operate on the two CSV files supplied with this project.